In [3]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, pipeline
from pathlib import Path
import glob
import os
from transformers import CLIPProcessor, CLIPModel
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
# Carga de datos
# Esta celda descarga todos los datos, y los extrae
import gdown, zipfile

url = 'https://drive.google.com/drive/folders/1MvRYh3SudRabydH2eAAzRq2K5Os1j41-'
output_folder = 'recsys-pf'

gdown.download_folder(url, output=output_folder, quiet=False, use_cookies=False)
zip_search_pattern = os.path.join(output_folder, '*.zip')
zip_files = glob.glob(zip_search_pattern)
image_zip_path = zip_files[0]

with zipfile.ZipFile(image_zip_path, 'r') as zip_ref:
    zip_ref.extractall(output_folder)

Retrieving folder contents


Processing file 1_CZsjxiXc-LsiXjRpiuEsGCZdvq7i0au interaction.csv
Processing file 1R95OpzeRwkG0Ljc9MJqKJI3zMSwqItHV item_info.csv
Processing file 1P4kKxL5xgJ_ePj955gIEej78bsnk36M5 resized_images.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1_CZsjxiXc-LsiXjRpiuEsGCZdvq7i0au
To: /content/recsys-pf/interaction.csv
100%|██████████| 28.1M/28.1M [00:00<00:00, 41.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1R95OpzeRwkG0Ljc9MJqKJI3zMSwqItHV
To: /content/recsys-pf/item_info.csv
100%|██████████| 25.0M/25.0M [00:00<00:00, 246MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1P4kKxL5xgJ_ePj955gIEej78bsnk36M5
From (redirected): https://drive.google.com/uc?id=1P4kKxL5xgJ_ePj955gIEej78bsnk36M5&confirm=t&uuid=f7198a58-8852-45b0-b614-da1fd664f483
To: /content/recsys-pf/resized_images.zip
100%|██████████| 163M/163M [00:00<00:00, 265MB/s]
Download completed


In [ ]:
import pandas as pd
import numpy as np

interaction_df = pd.read_csv('Datos/interaction.csv')
item_info_df = pd.read_csv('Datos/item_info_resumen.csv')

Seleccionamos solo los items a los que les extrajimos features

In [7]:
valid_item_ids = item_info_df['item_id'].unique()
interaction_df = interaction_df[interaction_df['item_id'].isin(valid_item_ids)]

In [8]:
sampled_interaction_df = interaction_df.sort_values('timestamp')
total_rows = len(sampled_interaction_df)
train_split_idx = int(total_rows * 0.80)
val_split_idx = int(total_rows * 0.90)
train_df = sampled_interaction_df.iloc[:train_split_idx].copy()
val_df = sampled_interaction_df.iloc[train_split_idx:val_split_idx].copy()
test_df = sampled_interaction_df.iloc[val_split_idx:].copy()

## Recomendaciones por perfil de usuario

Método 1, se fusionan las features, se calcula un promedio de los items interactuados por cada usuario y luego se calcula la similitud coseno entre los items. Para acelerar el código se usa el producto de matrices en base de calcular la similitud de coseno item a item

In [24]:
def vl_clip_recommend(train_df):
    fused_features = {}

    # Cargar features de imagen y texto
    images_features = {}
    textual_features = {}
    rutas_img = glob.glob(os.path.join("Features", "images_batch_*.pt"))
    rutas_txt = glob.glob(os.path.join("Features", "text_batch_*.pt"))
    for ruta in rutas_img:
        lote_img = torch.load(ruta)
        images_features.update(lote_img)
    for ruta in rutas_txt:
        lote_txt = torch.load(ruta)
        textual_features.update(lote_txt)
    print(f"Se cargaron las features de {len(images_features)} items")

    # Fuse image + text features per item
    for item_id in images_features.keys():
        img_feat = images_features[item_id].flatten()
        txt_feat = textual_features[item_id].flatten()

        # Concatenation (different embedding spaces)
        fused = np.concatenate([img_feat, txt_feat])

        fused = fused / np.linalg.norm(fused)
        fused_features[item_id] = fused

    item_ids = list(images_features.keys())
    item_matrix = np.array([fused_features[iid] for iid in item_ids])
    # Create user profiles (mean of interacted item fused embeddings)
    user_profiles = {}
    user_history = {} 

    for user_id, group in train_df.groupby('user_id'):
        user_items = group['item_id'].tolist()
        user_history[user_id] = set(user_items)
        valid_items = [iid for iid in user_items if iid in fused_features]
        if len(valid_items) == 0:
            continue
        # Mean embedding
        user_vec = np.mean([fused_features[iid] for iid in valid_items], axis=0)
        # Normalize
        user_vec = user_vec / np.linalg.norm(user_vec)
        user_profiles[user_id] = user_vec

    print(f"Se crearon perfiles para {len(user_profiles)} usuarios")

    # MATRIZ GLOBAL DE USUARIOS Y SIMILITUD
    user_ids = list(user_profiles.keys())
    user_matrix = np.array([user_profiles[uid] for uid in user_ids])

    print("Calculando matriz de similitud global...")
    # El producto punto de dos matrices normalizadas es la similitud coseno
    sim_matrix = np.dot(user_matrix, item_matrix.T)

    # Generar listas de recomendación de 10
    top_n = 10
    recommendations = {}
    item_to_idx = {iid: idx for idx, iid in enumerate(item_ids)}

    print("Generando recomendaciones...")
    for i, user_id in tqdm(enumerate(user_ids), total=len(user_ids)):
        user_sims = sim_matrix[i].copy()
        vistos = user_history.get(user_id, set())
        idx_vistos = [item_to_idx[iid] for iid in vistos if iid in item_to_idx]
        user_sims[idx_vistos] = -np.inf
        if np.max(user_sims) == -np.inf:
            continue
        ranked_idx = np.argsort(user_sims)[::-1][:top_n]
        u_recommendations = [(item_ids[idx], float(user_sims[idx])) for idx in ranked_idx]
        recommendations[user_id] = u_recommendations

    return recommendations

In [14]:
rec_vl_clip = vl_clip_recommend(train_df)

Se cargaron las features de 15000 items
Se crearon perfiles para 42933 usuarios
Calculando matriz de similitud global...
Generando recomendaciones...


100%|██████████| 42933/42933 [00:25<00:00, 1683.41it/s]


In [15]:
rec_vl_clip

{'u10002670': [('i210406', 0.8759765625),
  ('i50540', 0.87060546875),
  ('i275612', 0.8701171875),
  ('i13382', 0.859375),
  ('i116494', 0.8583984375),
  ('i284726', 0.8583984375),
  ('i34368', 0.85693359375),
  ('i228110', 0.85693359375),
  ('i201796', 0.85546875),
  ('i258508', 0.85546875)],
 'u10003782': [('i337423', 0.802734375),
  ('i30016', 0.798828125),
  ('i115082', 0.79736328125),
  ('i302708', 0.79736328125),
  ('i166576', 0.794921875),
  ('i85766', 0.791015625),
  ('i129253', 0.7890625),
  ('i142107', 0.78857421875),
  ('i220556', 0.78759765625),
  ('i51583', 0.787109375)],
 'u10004338': [('i172572', 0.826171875),
  ('i130330', 0.826171875),
  ('i183666', 0.82568359375),
  ('i75188', 0.8232421875),
  ('i197847', 0.82177734375),
  ('i144923', 0.82080078125),
  ('i277626', 0.818359375),
  ('i83680', 0.818359375),
  ('i223879', 0.81787109375),
  ('i102338', 0.8173828125)],
 'u10004764': [('i218604', 0.8798828125),
  ('i25472', 0.85986328125),
  ('i335987', 0.84521484375),
  ('

In [16]:
# De práctico_métricas.ipynb
def precision_at_k(r, k):
    assert 1 <= k <= r.size
    return (np.asarray(r)[:k] != 0).mean()

def average_precision_at_k(r, k):
    r = np.asarray(r)
    score = 0.
    for i in range(min(k, r.size)):
        score += precision_at_k(r, i + 1)
    return score / k

def dcg_at_k(r, k):
    r = np.asarray(r)[:k]
    if r.size:
        return np.sum(np.subtract(np.power(2, r), 1) / np.log2(np.arange(2, r.size + 2)))
    return 0.

def idcg_at_k(k):
    return dcg_at_k(np.ones(k), k)

def ndcg_at_k(r, k, max_relevant):
    idcg = idcg_at_k(min(k, max_relevant))
    if not idcg:
        return 0.
    return dcg_at_k(r, k) / idcg

def recall_at_k(relevant_items, recommended_items, k):
    relevant_items = set(relevant_items)
    recommended_items = set(recommended_items[:k])
    intersection = relevant_items.intersection(recommended_items)
    recall = len(intersection) / len(relevant_items) if len(relevant_items) > 0 else 0
    return recall

def evaluate_model(y_true, y_pred, N):
    map_scores = 0.
    ndcg_scores = 0.
    recall_scores = 0.
    
    for true_items, pred_items in zip(y_true, y_pred):
        
        true_set = set(true_items)
        
        # el relevance vector
        r = [1 if item in true_set else 0 for item in pred_items]
        
        # MAP
        user_map = average_precision_at_k(r, N)
        map_scores += user_map
        # NDGC@N
        max_relevant = len(true_items)
        user_ndcg = ndcg_at_k(r, N, max_relevant)
        ndcg_scores += user_ndcg
        # Recall@N
        user_recall = recall_at_k(true_items, pred_items, N)
        recall_scores += user_recall
        
    cantidad = len(y_true)
    final_map = map_scores / cantidad
    final_ndcg = ndcg_scores / cantidad
    final_recall = recall_scores / cantidad
    
    print(f"Métricas de evaluación ranking (Top-{N}):")
    print(f"MAP@{N}:    {final_map:.5f}")
    print(f"nDCG@{N}:   {final_ndcg:.5f}")
    print(f"Recall@{N}: {final_recall:.5f}")
    
    return final_map, final_ndcg, final_recall

In [17]:
# evaluación métricas
test_dict = test_df.groupby('user_id')['item_id'].apply(list).to_dict()

y_true = []
y_pred = []

N = 5

for user_id, true_items in test_dict.items():
    if user_id in rec_vl_clip:
        lista_predicha = [item_id for item_id, score in rec_vl_clip[user_id]][:N]
        
        if len(lista_predicha) > 0:
            y_true.append(true_items)
            y_pred.append(lista_predicha)

print(f"Total de usuarios en Test: {len(test_dict)}")
print(f"Usuarios evaluados (presentes en Train y con recomendaciones): {len(y_true)}\n")

map_score, ndcg_score, recall_score = evaluate_model(y_true, y_pred, N)


Total de usuarios en Test: 12144
Usuarios evaluados (presentes en Train y con recomendaciones): 9938

Métricas de evaluación ranking (Top-5):
MAP@5:    0.00068
nDCG@5:   0.00144
Recall@5: 0.00193


## Recomendaciones con lightFM

Unimos las features e interacciones para lightFM

In [25]:
import scipy.sparse as sp
from lightfm import LightFM

def vl_clip_lightfm(train_df, test_df, top_n):
    fused_features = {}

    # Cargar features de imagen y texto, lo mismo de antes
    images_features = {}
    textual_features = {}
    rutas_img = glob.glob(os.path.join("Features", "images_batch_*.pt"))
    rutas_txt = glob.glob(os.path.join("Features", "text_batch_*.pt"))
    for ruta in rutas_img:
        lote_img = torch.load(ruta)
        images_features.update(lote_img)
    for ruta in rutas_txt:
        lote_txt = torch.load(ruta)
        textual_features.update(lote_txt)
    print(f"Se cargaron las features de {len(images_features)} items")

    # Fuse image + text features per item
    for item_id in images_features.keys():
        img_feat = images_features[item_id].flatten()
        txt_feat = textual_features[item_id].flatten()

        # Concatenation (different embedding spaces)
        fused = np.concatenate([img_feat, txt_feat])

        fused = fused / np.linalg.norm(fused)
        fused_features[item_id] = fused

    todos_usuarios = pd.concat([train_df['user_id'], test_df['user_id']]).unique()
    todos_items = list(fused_features.keys())    
    user_id_map = {u: i for i, u in enumerate(todos_usuarios)}
    item_id_map = {item: i for i, item in enumerate(todos_items)}
    num_users = len(user_id_map)
    num_items = len(item_id_map)
    
    print(f"Total Usuarios: {num_users}\nTotal Ítems: {num_items}")

    # MATRIZ DE INTERACCIONES 
    train_rows = train_df['user_id'].map(user_id_map).values
    train_cols = train_df['item_id'].map(item_id_map).values
    train_data = np.ones(len(train_rows))
    
    train_interactions = sp.coo_matrix(
        (train_data, (train_rows, train_cols)), 
        shape=(num_users, num_items)
    )

    # MATRIZ DE FEATURES LATERALES
    feature_dim = len(next(iter(fused_features.values())))
    
    # Concatenamos una matriz identidad con las features.
    identidad = sp.identity(num_items)
    features_array = np.zeros((num_items, feature_dim))
    for item_id, idx in item_id_map.items():
        features_array[idx] = fused_features[item_id]
        
    features_sparse = sp.csr_matrix(features_array)
    item_features_matrix = sp.hstack([identidad, features_sparse])

    # ENTRENAR LIGHTFM
    modelo = LightFM(loss='warp', no_components=64, learning_rate=0.05, random_state=42)
    modelo.fit(
        train_interactions, 
        item_features=item_features_matrix, 
        epochs=20, 
        num_threads=4, 
        verbose=True
    )

    # GENERAR PREDICCIONES
    test_dict = test_df.groupby('user_id')['item_id'].apply(list).to_dict()
    train_history = train_df.groupby('user_id')['item_id'].apply(set).to_dict()
    
    y_true = []
    y_pred = []
    idx_a_item = {idx: item for item, idx in item_id_map.items()}

    _, user_embeddings = modelo.get_user_representations()
    _, item_embeddings = modelo.get_item_representations(features=item_features_matrix)
    
    # Producto punto globa
    sim_matrix = np.dot(user_embeddings, item_embeddings.T)

    for user_id, true_items in tqdm(test_dict.items(), total=len(test_dict), desc="Generando reankings"):
        if user_id not in user_id_map:
            continue 
            
        u_idx = user_id_map[user_id]

        scores = sim_matrix[u_idx].copy()
        vistos = train_history.get(user_id, set())
        vistos_idx = [item_id_map[iid] for iid in vistos if iid in item_id_map]
        scores[vistos_idx] = -np.inf
        
        top_n_idx = np.argsort(scores)[::-1][:top_n]
        lista_predicha = [idx_a_item[idx] for idx in top_n_idx]
        
        y_true.append(true_items)
        y_pred.append(lista_predicha)
        
    return y_true, y_pred


In [26]:
y_true_fm, y_pred_fm = vl_clip_lightfm(train_df, test_df, top_n=10)

Se cargaron las features de 15000 items
Total Usuarios: 45139
Total Ítems: 15000


KeyboardInterrupt: 

In [21]:
map_score_fm, ndcg_score_fm, recall_score_fm = evaluate_model(y_true_fm, y_pred_fm, N=5)

Métricas de evaluación ranking (Top-5):
MAP@5:    0.00021
nDCG@5:   0.00044
Recall@5: 0.00063


In [22]:
print(map_score, ndcg_score, recall_score)
print(map_score_fm, ndcg_score_fm, recall_score_fm)

0.0006808881733413833 0.0014445587844409717 0.0019314192437557298
0.0002143719806763285 0.00044102018962658313 0.0006335301172257694


In [23]:
evaluate_model(y_true, y_pred, N=10)
evaluate_model(y_true_fm, y_pred_fm, N=10)

Métricas de evaluación ranking (Top-10):
MAP@10:    0.00034
nDCG@10:   0.00143
Recall@10: 0.00193
Métricas de evaluación ranking (Top-10):
MAP@10:    0.00023
nDCG@10:   0.00077
Recall@10: 0.00157


(0.00023301401698140833, 0.0007689621489058718, 0.001566534840276259)